# Notebook 3: Baseline Models (LightGBM LambdaMART)
    
We train a Learning-to-Rank model using engineered features to establish our baseline.


In [1]:
!pip install lightgbm --quiet

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path(r"C:\Users\Acer\OneDrive - National University of Singapore\BT4222\JDsearch\processed")
cand_df = pd.read_parquet(PROCESSED_DIR / "candidates_split.parquet")
hist_df = pd.read_parquet(PROCESSED_DIR / "history_split.parquet")

print("Engineering basic features...")
# 1. Item Popularity (TRAIN history only)
train_hist = hist_df[hist_df.split == "train"]
pop_dict = train_hist["item_id"].value_counts().to_dict()
cand_df["item_pop"] = cand_df["item_id"].map(pop_dict).fillna(0)

# 2. User History Length (TRAIN history only)
hist_counts = train_hist.groupby("session_id").size().to_dict()
cand_df["hist_len"] = cand_df["session_id"].map(hist_counts).fillna(0)

# 3. Slate Size
slate_sizes = cand_df.groupby("session_id").size().to_dict()
cand_df["slate_size"] = cand_df["session_id"].map(slate_sizes)

print("Features ready.")
print(cand_df[["session_id", "item_id", "item_pop", "hist_len", "slate_size"]].head())

Engineering basic features...
Features ready.
   session_id   item_id  item_pop  hist_len  slate_size
0           0  36098226       0.0       0.0          90
1           0  17729172       0.0       0.0          90
2           0  34921664       0.0       0.0          90
3           0  68317377       0.0       0.0          90
4           0    693764       4.0       0.0          90


## Train LambdaMART

In [3]:
import lightgbm as lgb

def group_sizes(df):
    return df.groupby("session_id").size().values

train_df = cand_df[cand_df.split == "train"].sort_values("session_id")
val_df = cand_df[cand_df.split == "val"].sort_values("session_id")
test_df = cand_df[cand_df.split == "test"].sort_values("session_id")

features = ["item_pop", "hist_len", "slate_size"]

train_data = lgb.Dataset(train_df[features], label=train_df["raw_action_label"], group=group_sizes(train_df))
val_data = lgb.Dataset(val_df[features], label=val_df["raw_action_label"], group=group_sizes(val_df), reference=train_data)

params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'ndcg_eval_at': [10],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

print("Training LightGBM...")
model = lgb.train(params, train_data, num_boost_round=100, valid_sets=[val_data])
print("Training complete.")

Training LightGBM...
Training complete.


## Evaluate on Test Set

In [4]:
from sklearn.metrics import ndcg_score

test_df = test_df.copy()
test_df["lgb_score"] = model.predict(test_df[features])

ndcg_scores = []
for sid, group in test_df.groupby("session_id"):
    if (group["raw_action_label"] > 0).sum() > 0:
        if len(group) >= 2: 
            true_rel = group["raw_action_label"].values.reshape(1, -1)
            pred_rel = group["lgb_score"].values.reshape(1, -1)
            ndcg = ndcg_score(true_rel, pred_rel, k=10)
            ndcg_scores.append(ndcg)

print(f"Mean NDCG@10 on Test Set: {np.mean(ndcg_scores):.4f}")
print(f"Evaluated on {len(ndcg_scores)} sessions with positive labels.")

Mean NDCG@10 on Test Set: 0.3375
Evaluated on 25706 sessions with positive labels.
